# Testing 4-bit trigger codes w/ arduino & pyserial

In [ ]:
import serial
import time

import numpy as np

import matplotlib.pyplot as plt


import brainflow
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BrainFlowError, BoardIds
from brainflow_stream import BrainFlowBoardSetup

### CONFIG

In [ ]:
# list available serial ports
def list_serial_ports():
    import serial.tools.list_ports
    ports = serial.tools.list_ports.comports()
    print([port.device for port in ports])

In [ ]:
# CHANGE THIS to match your Arduino port:
#   Windows: "COM3", "COM4", ...
#   macOS:   "/dev/tty.usbmodemXXXX" or "/dev/tty.usbserialXXXX"
#   Linux:   "/dev/ttyACM0", "/dev/ttyUSB0", etc.
list_serial_ports()


SERIAL_PORT = "COM5" #"/dev/ttyACM0"
BAUD_RATE = 115200


## OpenBCI Connection

In [ ]:
board_id = BoardIds.CYTON_BOARD.value # Set the board_id to match the Cyton board

# Lets quickly take a look at the specifications of the Cyton board
for item1, item2 in BoardShim.get_board_descr(board_id).items():
    print(f"{item1}: {item2}")

In [ ]:
cyton_board = BrainFlowBoardSetup(
                                board_id = board_id,
                                name = 'Board_1',   # Optional name for the board. This is useful if you have multiple boards connected and want to distinguish between them.
                                serial_port = None  # If the serial port is not specified, it will try to auto-detect the board. If this fails, you will have to assign the correct serial port. See https://docs.openbci.com/GettingStarted/Boards/CytonGS/ 
                                ) 

cyton_board.setup() # This will establish a connection to the board and start streaming data.

In [ ]:
board_info = cyton_board.get_board_info() # Retrieves the EEG channel and sampling rate of the board.
print(f"Board info: {board_info}")

board_srate = cyton_board.get_sampling_rate() # Retrieves the sampling rate of the board.
print(f"Board sampling rate: {board_srate}")

In [ ]:

# raw_data_all = cyton_board.get_board_data()

## Functions

In [ ]:

def open_trigger_serial(port=SERIAL_PORT, baud_rate=BAUD_RATE):
    ser = serial.Serial(port, baud_rate, timeout=1)
    # Arduino auto-resets when serial opens; give it a moment
    time.sleep(2.0)
    
    print(f"Opened serial port {port} at {baud_rate} baud.")
    
    return ser


def send_trigger(ser, code: int, duration_ms: int):
    """
    Send a trigger code (0–15) and pulse duration (ms, 0–65535) to Arduino.
    """
    if not (0 <= code <= 15):
        raise ValueError("code must be in 0–15")

    if not (0 <= duration_ms <= 65535):
        raise ValueError("duration_ms must be in 0–65535")

    low  = duration_ms & 0xFF
    high = (duration_ms >> 8) & 0xFF

    packet = bytes([code, low, high])
    
    # Clear any stale data first
    ser.reset_input_buffer()
    
    ser.write(packet)
    ser.flush()


## Explanation:
OpenBCI board has 4 input pins resulting in 16 (2^4) possible trigger codes (0-15). Each code is represented by the binary state of the 4 pins. For example:
- Code 0: 0000 (all pins LOW)
- Code 1: 0001 (pin 1 HIGH, others LOW)
- Code 15: 1111 (all pins HIGH)

The arduino is programmed to take an input code (0-15) via serial and set the corresponding pins HIGH/LOW based on the binary representation of that code.


## Tests

In [ ]:
TRIG_DURATION_MS = 200

# Pull data to clear the buffer
raw_data_all = cyton_board.get_board_data()
print(raw_data_all.shape)

ser = open_trigger_serial()
time.sleep(3)  # small delay for Arduino reset

for _ in range(2):
    for code in range(1, 16):
        print(f"Sending trigger code: {code} -> {code:04b}") # (bytes: {bytes([code])})")
        send_trigger(ser, code, duration_ms=TRIG_DURATION_MS)
        time.sleep(TRIG_DURATION_MS/1000 + 0.5)  # wait for pulse + small gap
        
    time.sleep(2) 
        
ser.close()

# Pull data after trigger codes sent!
raw_data_all = cyton_board.get_board_data()
print(raw_data_all.shape)

# Check Data

In [ ]:
raw_data_all.shape

In [ ]:
# ===== SPECIFY WHICH CHANNELS TO PLOT =====
# Option 1: Plot a range of channels
CHANNELS_TO_PLOT = range(0, 24)  # All channels
# CHANNELS_TO_PLOT = range(13, 21)  # Just channels 12-15 (suspected digital)
# CHANNELS_TO_PLOT = range(0, 8)    # Just EEG channels
# CHANNELS_TO_PLOT = [14, 15, 16, 17, 18, 19, 20]  # Specific individual channels
# CHANNELS_TO_PLOT = [8, 9, 10, 11, 12, 13, 14, 15, 16, 17]  # Multiple ranges

channels_list = list(CHANNELS_TO_PLOT)
num_channels = len(channels_list)

# Plot selected channels
fig, axes = plt.subplots(num_channels, 1, figsize=(16, 3*num_channels))
if num_channels == 1:
    axes = [axes]  # Handle single subplot case

fig.suptitle(f'Channels {channels_list[0]}-{channels_list[-1]} from Cyton Board', fontsize=16, fontweight='bold')

sampling_rate = cyton_board.get_sampling_rate()
time_seconds = np.arange(raw_data_all.shape[1]) / sampling_rate

for plot_idx, ch_idx in enumerate(channels_list):
    axes[plot_idx].plot(time_seconds, raw_data_all[ch_idx, :], linewidth=0.5, color='steelblue')
    axes[plot_idx].set_ylabel(f'Ch {ch_idx}', fontsize=10, fontweight='bold')
    axes[plot_idx].grid(True, alpha=0.2)
    axes[plot_idx].margins(0)

axes[-1].set_xlabel('Time (seconds)', fontsize=10)
plt.tight_layout()
plt.show()

# Print statistics for selected channels
print("\nChannel Statistics:")
print("-" * 70)
for ch_idx in channels_list:
    min_val = raw_data_all[ch_idx, :].min()
    max_val = raw_data_all[ch_idx, :].max()
    mean_val = raw_data_all[ch_idx, :].mean()
    std_val = raw_data_all[ch_idx, :].std()
    print(f"Ch {ch_idx:2d}: min={min_val:8.2f}, max={max_val:8.2f}, mean={mean_val:8.2f}, std={std_val:6.2f}")

In [ ]:
def test_all_combinations(data, channel_range=range(13, 21), target_codes=None):
    """
    Test ALL possible 4-channel combinations to find the correct digital channels.
    """
    from itertools import combinations
    
    if target_codes is None:
        target_codes = set(range(1, 16))  # codes 1-15
    
    results = []
    
    # Generate all possible 4-channel combinations
    all_combos = list(combinations(channel_range, 4))
    print(f"Testing {len(all_combos)} combinations...\n")
    
    for combo in all_combos:
        digital_data = data[list(combo), :]
        
        # Try both bit orderings (LSB-first and MSB-first)
        for bit_order in ['LSB_first', 'MSB_first']:
            trigger_codes = []
            for sample_idx in range(digital_data.shape[1]):
                bits = [int(digital_data[i, sample_idx] > 0.5) for i in range(4)]
                
                if bit_order == 'LSB_first':
                    code = bits[0] + (bits[1] << 1) + (bits[2] << 2) + (bits[3] << 3)
                else:  # MSB_first
                    code = (bits[0] << 3) + (bits[1] << 2) + (bits[2] << 1) + bits[3]
                
                trigger_codes.append(code)
            
            trigger_codes = np.array(trigger_codes)
            non_zero = trigger_codes[trigger_codes > 0]
            unique_codes = set(np.unique(non_zero))
            
            # Score: how many of the expected codes did we find?
            match_score = len(unique_codes & target_codes)
            
            if match_score >= 12:  # At least 12 of the 15 codes found
                results.append({
                    'channels': combo,
                    'bit_order': bit_order,
                    'codes_found': sorted(list(unique_codes)),
                    'match_score': match_score
                })
    
    # Sort by match score (best first)
    results.sort(key=lambda x: x['match_score'], reverse=True)
    
    # Display top 10 results
    print("Top matches:\n")
    for i, result in enumerate(results[:10]):
        print(f"{i+1}. Channels {result['channels']} ({result['bit_order']})")
        print(f"   Found {result['match_score']}/15 codes: {result['codes_found']}")
        print()
    
    return results

# Search through all channels
all_results = test_all_combinations(raw_data_all, channel_range=range(10, 24))